In [1]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
import tensorflow as tf

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU: {tf.config.list_physical_devices('GPU')}")

# Data load
print("\nData load ho raha hai...")
parsed_path = "/content/drive/MyDrive/ASAD_Thesis/processed_data/spark_logs_parsed.csv"
df = pd.read_csv(parsed_path)
print(f"Shape: {df.shape}")

# Features
le_app = LabelEncoder()
le_container = LabelEncoder()
df['app_id_enc'] = le_app.fit_transform(df['app_id'])
df['container_id_enc'] = le_container.fit_transform(df['container_id'])
df['binary_label'] = df['label'].apply(lambda x: 0 if x == 0 else 1)

features = ['cluster_id', 'app_id_enc', 'container_id_enc']
X = df[features].values
y = df['binary_label'].values

# Scale karo
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y)

print(f"\nX_train: {X_train.shape}")
print(f"X_test: {X_test.shape}")
print(f"Train anomalies: {(y_train==1).sum():,}")
print(f"Test anomalies: {(y_test==1).sum():,}")
print("\nSab ready hai! ✅")

Mounted at /content/drive
TensorFlow version: 2.20.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

Data load ho raha hai...
Shape: (1661830, 6)

X_train: (1329464, 3)
X_test: (332366, 3)
Train anomalies: 1,820
Test anomalies: 455

Sab ready hai! ✅


In [2]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, RepeatVector, TimeDistributed, Dense
from tensorflow.keras.callbacks import EarlyStopping
import numpy as np

print("LSTM Autoencoder banana shuru...")

# LSTM ke liye 3D reshape karo (samples, timesteps, features)
X_train_lstm = X_train.reshape((X_train.shape[0], 1, X_train.shape[1]))
X_test_lstm = X_test.reshape((X_test.shape[0], 1, X_test.shape[1]))

# Sirf normal data pe train karo
X_train_normal = X_train_lstm[y_train == 0]
print(f"Normal training samples: {len(X_train_normal):,}")

# LSTM Autoencoder architecture
inputs = Input(shape=(1, 3))
encoded = LSTM(32, activation='relu', return_sequences=False)(inputs)
repeated = RepeatVector(1)(encoded)
decoded = LSTM(32, activation='relu', return_sequences=True)(repeated)
output = TimeDistributed(Dense(3))(decoded)

autoencoder = Model(inputs, output)
autoencoder.compile(optimizer='adam', loss='mse')
autoencoder.summary()

# Train
print("\nTraining shuru...")
early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

history = autoencoder.fit(
    X_train_normal, X_train_normal,
    epochs=20,
    batch_size=512,
    validation_split=0.1,
    callbacks=[early_stop],
    verbose=1
)

print("\nTraining complete! ✅")

LSTM Autoencoder banana shuru...
Normal training samples: 1,327,644


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 1, 3)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 32)             │         4,608 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ repeat_vector (RepeatVector)    │ (None, 1, 32)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 1, 32)          │         8,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed                │ (None, 1, 3)           │            99 │
│ (TimeDistributed)               │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 13,027 (50.89 KB)

 Trainable params: 13,027 (50.89 KB)

 Non-trainable params: 0 (0.00 B)


Training shuru...
Epoch 1/20
2334/2334 ━━━━━━━━━━━━━━━━━━━━ 15s 4ms/step - loss: 0.0327 - val_loss: 1.8234e-04
Epoch 2/20
2334/2334 ━━━━━━━━━━━━━━━━━━━━ 8s 3ms/step - loss: 7.9215e-05 - val_loss: 2.8789e-05
Epoch 3/20
2334/2334 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - loss: 1.6071e-05 - val_loss: 8.4794e-06
Epoch 4/20
2334/2334 ━━━━━━━━━━━━━━━━━━━━ 8s 3ms/step - loss: 6.8254e-06 - val_loss: 6.6685e-06
Epoch 5/20
2334/2334 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - loss: 4.8209e-06 - val_loss: 3.7622e-06
Epoch 6/20
2334/2334 ━━━━━━━━━━━━━━━━━━━━ 8s 3ms/step - loss: 3.9410e-06 - val_loss: 2.8257e-06
Epoch 7/20
2334/2334 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - loss: 3.3594e-06 - val_loss: 2.0493e-06
Epoch 8/20
2334/2334 ━━━━━━━━━━━━━━━━━━━━ 8s 3ms/step - loss: 2.8276e-06 - val_loss: 2.8424e-06
Epoch 9/20
2334/2334 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - loss: 2.8539e-06 - val_loss: 2.2538e-06
Epoch 10/20
2334/2334 ━━━━━━━━━━━━━━━━━━━━ 8s 3ms/step - loss: 2.6235e-06 - val_loss: 1.2915e-06
Epoch 11/20
2334/2334 ━

In [3]:
import numpy as np
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

print("Reconstruction error calculate kar rahe hain...")

# Test data reconstruct karo
X_test_pred = autoencoder.predict(X_test_lstm, batch_size=512, verbose=0)

# Reconstruction error (MSE per sample)
mse = np.mean(np.power(X_test_lstm - X_test_pred, 2), axis=(1, 2))

print(f"Normal MSE mean:  {mse[y_test==0].mean():.6f}")
print(f"Anomaly MSE mean: {mse[y_test==1].mean():.6f}")

# Threshold — 95th percentile of normal errors
threshold = np.percentile(mse[y_test==0], 95)
print(f"\nThreshold (95th percentile): {threshold:.6f}")

# Predict
y_pred_lstm = (mse > threshold).astype(int)

print("\n=== LSTM Autoencoder Results ===")
print(classification_report(y_test, y_pred_lstm,
                            target_names=['Normal', 'Anomaly']))

roc_auc = roc_auc_score(y_test, mse)
print(f"ROC-AUC Score: {roc_auc:.4f}")

cm = confusion_matrix(y_test, y_pred_lstm)
print(f"\nConfusion Matrix:")
print(cm)

Reconstruction error calculate kar rahe hain...
Normal MSE mean:  0.000001
Anomaly MSE mean: 0.000114

Threshold (95th percentile): 0.000002

=== LSTM Autoencoder Results ===
              precision    recall  f1-score   support

      Normal       1.00      0.95      0.97    331911
     Anomaly       0.02      0.93      0.05       455

    accuracy                           0.95    332366
   macro avg       0.51      0.94      0.51    332366
weighted avg       1.00      0.95      0.97    332366

ROC-AUC Score: 0.9822

Confusion Matrix:
[[315349  16562]
 [    31    424]]


In [4]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from imblearn.over_sampling import SMOTE
import numpy as np

print("CNN Model banana shuru...")

# SMOTE for CNN (supervised)
print("SMOTE apply ho raha hai...")
smote = SMOTE(random_state=42, k_neighbors=5)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

# CNN ke liye 3D reshape
X_train_cnn = X_train_sm.reshape((X_train_sm.shape[0], 3, 1))
X_test_cnn = X_test.reshape((X_test.shape[0], 3, 1))

print(f"X_train_cnn: {X_train_cnn.shape}")
print(f"X_test_cnn: {X_test_cnn.shape}")

# CNN Architecture
model_cnn = Sequential([
    Conv1D(filters=32, kernel_size=2, activation='relu', input_shape=(3, 1)),
    Dropout(0.2),
    Conv1D(filters=64, kernel_size=1, activation='relu'),
    Flatten(),
    Dense(32, activation='relu'),
    Dropout(0.2),
    Dense(1, activation='sigmoid')
])

model_cnn.compile(optimizer='adam',
                  loss='binary_crossentropy',
                  metrics=['accuracy'])
model_cnn.summary()

# Train
print("\nTraining shuru...")
early_stop = EarlyStopping(monitor='val_loss', patience=3,
                            restore_best_weights=True)

history_cnn = model_cnn.fit(
    X_train_cnn, y_train_sm,
    epochs=20,
    batch_size=512,
    validation_split=0.1,
    callbacks=[early_stop],
    verbose=1
)

print("\nCNN Training complete! ✅")

CNN Model banana shuru...
SMOTE apply ho raha hai...
X_train_cnn: (2655288, 3, 1)
X_test_cnn: (332366, 3, 1)


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 2, 32)          │            96 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 2, 32)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 2, 64)          │         2,112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         4,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,369 (24.88 KB)

 Trainable params: 6,369 (24.88 KB)

 Non-trainable params: 0 (0.00 B)


Training shuru...
Epoch 1/20
4668/4668 ━━━━━━━━━━━━━━━━━━━━ 24s 4ms/step - accuracy: 0.9569 - loss: 0.1195 - val_accuracy: 0.9935 - val_loss: 0.0470
Epoch 2/20
4668/4668 ━━━━━━━━━━━━━━━━━━━━ 16s 3ms/step - accuracy: 0.9794 - loss: 0.0635 - val_accuracy: 0.9925 - val_loss: 0.0282
Epoch 3/20
4668/4668 ━━━━━━━━━━━━━━━━━━━━ 15s 3ms/step - accuracy: 0.9857 - loss: 0.0518 - val_accuracy: 0.9980 - val_loss: 0.0209
Epoch 4/20
4668/4668 ━━━━━━━━━━━━━━━━━━━━ 16s 3ms/step - accuracy: 0.9879 - loss: 0.0459 - val_accuracy: 0.9984 - val_loss: 0.0186
Epoch 5/20
4668/4668 ━━━━━━━━━━━━━━━━━━━━ 16s 3ms/step - accuracy: 0.9886 - loss: 0.0429 - val_accuracy: 0.9990 - val_loss: 0.0142
Epoch 6/20
4668/4668 ━━━━━━━━━━━━━━━━━━━━ 17s 4ms/step - accuracy: 0.9890 - loss: 0.0413 - val_accuracy: 0.9988 - val_loss: 0.0187
Epoch 7/20
4668/4668 ━━━━━━━━━━━━━━━━━━━━ 16s 3ms/step - accuracy: 0.9895 - loss: 0.0394 - val_accuracy: 0.9992 - val_loss: 0.0109
Epoch 8/20
4668/4668 ━━━━━━━━━━━━━━━━━━━━ 15s 3ms/step - accurac

In [5]:
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
import numpy as np

print("CNN Evaluation...")

# Predict
y_prob_cnn = model_cnn.predict(X_test_cnn, batch_size=512, verbose=0).flatten()
y_pred_cnn = (y_prob_cnn > 0.5).astype(int)

print("\n=== CNN Results ===")
print(classification_report(y_test, y_pred_cnn,
                            target_names=['Normal', 'Anomaly']))

roc_auc = roc_auc_score(y_test, y_prob_cnn)
print(f"ROC-AUC Score: {roc_auc:.4f}")

cm = confusion_matrix(y_test, y_pred_cnn)
print(f"\nConfusion Matrix:")
print(cm)

# ============================================
# FINAL COMPARISON - Sab models
# ============================================
print("\n" + "=" * 55)
print("DEEP LEARNING vs TRADITIONAL ML — FULL COMPARISON")
print("=" * 55)

comparison = {
    "Random Forest":      {"ROC_AUC": 0.9902, "Recall": 0.89, "F1": 0.44},
    "Isolation Forest":   {"ROC_AUC": 0.9559, "Recall": 0.04, "F1": 0.04},
    "One-Class SVM":      {"ROC_AUC": 0.2388, "Recall": 0.01, "F1": 0.00},
    "LSTM Autoencoder":   {"ROC_AUC": 0.9822, "Recall": 0.93, "F1": 0.05},
    "CNN Classifier":     {"ROC_AUC": roc_auc, "Recall": None, "F1": None},
}

print(f"\n{'Model':<22} {'ROC-AUC':>10} {'Recall':>10} {'F1':>10}")
print("-" * 55)
for model, m in comparison.items():
    recall = f"{m['Recall']:.2f}" if m['Recall'] is not None else "TBD"
    f1 = f"{m['F1']:.2f}" if m['F1'] is not None else "TBD"
    print(f"{model:<22} {m['ROC_AUC']:>10.4f} {recall:>10} {f1:>10}")

CNN Evaluation...

=== CNN Results ===
              precision    recall  f1-score   support

      Normal       1.00      0.99      0.99    331911
     Anomaly       0.09      1.00      0.16       455

    accuracy                           0.99    332366
   macro avg       0.54      0.99      0.58    332366
weighted avg       1.00      0.99      0.99    332366

ROC-AUC Score: 0.9978

Confusion Matrix:
[[327261   4650]
 [     2    453]]

DEEP LEARNING vs TRADITIONAL ML — FULL COMPARISON

Model                     ROC-AUC     Recall         F1
-------------------------------------------------------
Random Forest              0.9902       0.89       0.44
Isolation Forest           0.9559       0.04       0.04
One-Class SVM              0.2388       0.01       0.00
LSTM Autoencoder           0.9822       0.93       0.05
CNN Classifier             0.9978        TBD        TBD


In [6]:
import json, os

save_dir = "/content/drive/MyDrive/ASAD_Thesis/results"
os.makedirs(save_dir, exist_ok=True)

# Models save karo
autoencoder.save(f"{save_dir}/lstm_autoencoder.keras")
model_cnn.save(f"{save_dir}/cnn_classifier.keras")
print("Models saved! ✅")

# Results save
full_results = {
    "Traditional_ML": {
        "Random_Forest":    {"ROC_AUC": 0.9902, "Recall": 0.89, "F1": 0.44},
        "Isolation_Forest": {"ROC_AUC": 0.9559, "Recall": 0.04, "F1": 0.04},
        "OneClass_SVM":     {"ROC_AUC": 0.2388, "Recall": 0.01, "F1": 0.00}
    },
    "Deep_Learning": {
        "LSTM_Autoencoder": {"ROC_AUC": 0.9822, "Recall": 0.93, "F1": 0.05},
        "CNN_Classifier":   {"ROC_AUC": 0.9978, "Recall": 1.00, "F1": 0.16}
    }
}

with open(f"{save_dir}/all_results.json", "w") as f:
    json.dump(full_results, f, indent=4)

print("Results saved! ✅")
print(json.dumps(full_results, indent=4))

Models saved! ✅
Results saved! ✅
{
    "Traditional_ML": {
        "Random_Forest": {
            "ROC_AUC": 0.9902,
            "Recall": 0.89,
            "F1": 0.44
        },
        "Isolation_Forest": {
            "ROC_AUC": 0.9559,
            "Recall": 0.04,
            "F1": 0.04
        },
        "OneClass_SVM": {
            "ROC_AUC": 0.2388,
            "Recall": 0.01,
            "F1": 0.0
        }
    },
    "Deep_Learning": {
        "LSTM_Autoencoder": {
            "ROC_AUC": 0.9822,
            "Recall": 0.93,
            "F1": 0.05
        },
        "CNN_Classifier": {
            "ROC_AUC": 0.9978,
            "Recall": 1.0,
            "F1": 0.16
        }
    }
}
